# 🧠 SynthRAD2023: Clinical Brain MRI-to-CT Diffusion Model
## SOTA Architectural & Loss Overhaul: Attention Gates, Bone-Weighted Loss & DDIM
### Target Performance Benchmarks:
- **Test SSIM:** $\ge \mathbf{0.8500}$ (Up from $0.7516$)
- **Test PSNR:** $\ge \mathbf{26.0\text{ dB}}$ (Up from $20.65\text{ dB}$)
- **Test HU-MAE:** $\le \mathbf{65.0\text{ HU}}$ (Down from $113.3\text{ HU}$)
- **Bone Dice (HU > 300):** $\ge \mathbf{0.8500}$ (Up from $0.7812$)
- **Bone HD95:** $\le \mathbf{4.5\text{ mm}}$ (Down from $10.15\text{ mm}$)


In [ ]:
import os
import glob
import math
import random
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from skimage.metrics import structural_similarity as compute_ssim
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from scipy.spatial.distance import directed_hausdorff

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("[OK] Deterministic seed locked (SEED=42).")


In [ ]:
def extract_body_contour_mask(img_arr, threshold=15):
    """Extracts clean solid head contour, zeroing exterior scanner air."""
    binary = (img_arr > threshold).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(img_arr, dtype=np.uint8)
    if contours:
        largest_c = max(contours, key=cv2.contourArea)
        cv2.drawContours(mask, [largest_c], -1, 255, thickness=cv2.FILLED)
    else:
        mask = closed
    return (mask > 0).astype(np.float32)

class SynthRADClinicalDataset(Dataset):
    """
    3-Channel Input: [x_t, mri, mask]
    Applies Otsu body contour masking, physical [-1, 1] scaling, and synchronous augmentations.
    """
    def __init__(self, mri_paths, ct_paths, image_size=128, is_train=False):
        self.mri_paths = mri_paths
        self.ct_paths  = ct_paths
        self.image_size = image_size
        self.is_train = is_train

    def __len__(self):
        return len(self.mri_paths)

    def _process(self, path):
        img = Image.open(path).convert('L')
        arr = np.array(img, dtype=np.float32)
        mask = extract_body_contour_mask(arr)
        arr = arr * mask
        
        arr_res  = cv2.resize(arr, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        mask_res = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        
        norm = (arr_res / 255.0) * 2.0 - 1.0
        norm[mask_res < 0.5] = -1.0
        return torch.from_numpy(norm).unsqueeze(0).float(), torch.from_numpy(mask_res).unsqueeze(0).float()

    def __getitem__(self, idx):
        mri, m_mask = self._process(self.mri_paths[idx])
        ct,  c_mask = self._process(self.ct_paths[idx])
        union_mask = torch.clamp(m_mask + c_mask, 0.0, 1.0)

        if self.is_train:
            if random.random() > 0.5:
                mri = TF.hflip(mri)
                ct  = TF.hflip(ct)
                union_mask = TF.hflip(union_mask)
            angle = random.uniform(-8.0, 8.0)
            tx, ty = int(random.uniform(-4, 4)), int(random.uniform(-4, 4))
            scale = random.uniform(0.96, 1.04)
            mri = TF.affine(mri, angle=angle, translate=[tx, ty], scale=scale, shear=0, fill=-1.0)
            ct  = TF.affine(ct,  angle=angle, translate=[tx, ty], scale=scale, shear=0, fill=-1.0)
            union_mask = TF.affine(union_mask, angle=angle, translate=[tx, ty], scale=scale, shear=0, fill=0.0)

        return {"mri": mri, "ct": ct, "mask": union_mask, "mri_path": self.mri_paths[idx], "ct_path": self.ct_paths[idx]}

# Discover Dataset & Split
base_dir = "/kaggle/input/datasets/lohithkgowda/synthrad2023-brain-2d-png/synthrad_png"
if not os.path.exists(base_dir):
    for c in glob.glob("/kaggle/input/**/synthrad_png", recursive=True):
        if os.path.exists(os.path.join(c, "ct")):
            base_dir = c
            break

ct_dir, mri_dir = os.path.join(base_dir, "ct"), os.path.join(base_dir, "mri")
paired = sorted(list(set(os.listdir(ct_dir)).intersection(set(os.listdir(mri_dir)))))
pat_map = {}
for f in paired:
    pat_map.setdefault(f.split("_")[0], []).append(f)

patients = sorted(list(pat_map.keys()))
rng = np.random.RandomState(SEED)
perm = rng.permutation(patients).tolist()

n_tr = int(round(len(patients) * 0.80))
n_va = int(round(len(patients) * 0.10))
train_p, val_p, test_p = sorted(perm[:n_tr]), sorted(perm[n_tr:n_tr+n_va]), sorted(perm[n_tr+n_va:])

def get_paths(plist):
    m, c = [], []
    for p in plist:
        for f in sorted(pat_map[p]):
            m.append(os.path.join(mri_dir, f))
            c.append(os.path.join(ct_dir, f))
    return m, c

tr_m, tr_c = get_paths(train_p)
va_m, va_c = get_paths(val_p)
te_m, te_c = get_paths(test_p)

train_loader = DataLoader(SynthRADClinicalDataset(tr_m, tr_c, is_train=True), batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(SynthRADClinicalDataset(va_m, va_c, is_train=False), batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SynthRADClinicalDataset(te_m, te_c, is_train=False), batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders Ready: Train={len(tr_m)} slices ({len(train_p)} pts) | Val={len(va_m)} slices ({len(val_p)} pts) | Test={len(te_m)} slices ({len(test_p)} pts)")


In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000.0) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class AttentionGate(nn.Module):
    """Suppresses background noise and highlights skull/brain boundaries."""
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.w_g = nn.Sequential(nn.Conv2d(f_g, f_int, 1), nn.GroupNorm(8, f_int))
        self.w_x = nn.Sequential(nn.Conv2d(f_l, f_int, 1), nn.GroupNorm(8, f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        return x * self.psi(self.relu(self.w_g(g) + self.w_x(x)))

class RegularizedResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, dropout=0.10):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.gn1 = nn.GroupNorm(8, out_ch)
        self.gn2 = nn.GroupNorm(8, out_ch)
        self.dropout = nn.Dropout2d(p=dropout)
        self.act = nn.SiLU()
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.act(self.gn1(self.conv1(x)))
        h = h + self.act(self.time_mlp(t_emb))[:, :, None, None]
        h = self.dropout(self.act(self.gn2(self.conv2(h))))
        return h + self.shortcut(x)

class ResAttentionUNet(nn.Module):
    """Attention-Gated Residual U-Net accepting 3 channels [x_t, mri, mask]."""
    def __init__(self, in_channels=3, out_channels=1, time_emb_dim=64, dropout=0.10):
        super().__init__()
        self.time_mlp = nn.Sequential(SinusoidalPositionEmbeddings(time_emb_dim), nn.Linear(time_emb_dim, time_emb_dim), nn.SiLU())
        self.enc1 = RegularizedResBlock(in_channels, 64, time_emb_dim, dropout)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = RegularizedResBlock(64, 128, time_emb_dim, dropout)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = RegularizedResBlock(128, 256, time_emb_dim, dropout)
        self.pool3 = nn.MaxPool2d(2)
        self.bot1 = RegularizedResBlock(256, 512, time_emb_dim, dropout)
        self.bot2 = RegularizedResBlock(512, 256, time_emb_dim, dropout)
        self.gate3 = AttentionGate(256, 256, 128)
        self.gate2 = AttentionGate(128, 128, 64)
        self.gate1 = AttentionGate(64,  64,  32)
        self.up3 = nn.ConvTranspose2d(256, 256, 2, stride=2)
        self.dec3 = RegularizedResBlock(256 + 256, 128, time_emb_dim, dropout)
        self.up2 = nn.ConvTranspose2d(128, 128, 2, stride=2)
        self.dec2 = RegularizedResBlock(128 + 128, 64, time_emb_dim, dropout)
        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec1 = RegularizedResBlock(64 + 64, 64, time_emb_dim, dropout)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        e1 = self.enc1(x, t_emb)
        e2 = self.enc2(self.pool1(e1), t_emb)
        e3 = self.enc3(self.pool2(e2), t_emb)
        b  = self.bot2(self.bot1(self.pool3(e3), t_emb), t_emb)
        u3 = self.up3(b)
        d3 = self.dec3(torch.cat([u3, self.gate3(u3, e3)], dim=1), t_emb)
        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, self.gate2(u2, e2)], dim=1), t_emb)
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, self.gate1(u1, e1)], dim=1), t_emb)
        return self.final(d1)

model = ResAttentionUNet(in_channels=3, out_channels=1, time_emb_dim=64, dropout=0.10).to(device)
print(f"[OK] ResAttentionUNet loaded. Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
class DDPMScheduler:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.timesteps = timesteps
        self.device = torch.device(device) if isinstance(device, str) else device
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(self.device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(self.device)

    def add_noise(self, x_0, t):
        noise = torch.randn_like(x_0)
        alphas_cp = self.alphas_cumprod.to(x_0.device)
        sqrt_alpha = alphas_cp[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus_alpha = (1.0 - alphas_cp[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_alpha * x_0 + sqrt_one_minus_alpha * noise, noise

    @torch.no_grad()
    def sample_ddim(self, model, mri_condition, mask_condition, ddim_steps=50):
        """DDIM 50-Step Deterministic Reverse Sampler (Runs in 1.5s/slice, eliminates noise drift)."""
        model.eval()
        device = mri_condition.device
        b = mri_condition.shape[0]
        x_t = torch.randn_like(mri_condition).to(device)
        timesteps = sorted(torch.linspace(0, self.timesteps - 1, ddim_steps).long().tolist(), reverse=True)
        alphas_cp = self.alphas_cumprod.to(device)

        for i, t_val in enumerate(timesteps):
            t = torch.full((b,), t_val, device=device, dtype=torch.long)
            pred_noise = model(torch.cat([x_t, mri_condition, mask_condition], dim=1), t)
            alpha_cur = alphas_cp[t_val]
            alpha_prev = alphas_cp[timesteps[i + 1]] if i < len(timesteps) - 1 else torch.tensor(1.0, device=device)
            pred_x0 = (x_t - (1.0 - alpha_cur).sqrt() * pred_noise) / (alpha_cur.sqrt() + 1e-8)
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)
            dir_xt = (1.0 - alpha_prev).clamp(min=0.0).sqrt() * pred_noise
            x_t = alpha_prev.sqrt() * pred_x0 + dir_xt

        x_t = torch.where(mask_condition > 0.5, x_t, -1.0)
        return torch.clamp(x_t, -1.0, 1.0)

def predict_x0(x_t, predicted_noise, t, scheduler):
    alphas_cumprod = scheduler.alphas_cumprod.to(x_t.device)
    sqrt_alpha_hat = alphas_cumprod[t].sqrt().view(-1, 1, 1, 1)
    sqrt_one_minus_alpha_hat = (1.0 - alphas_cumprod[t]).sqrt().view(-1, 1, 1, 1)
    return torch.clamp((x_t - sqrt_one_minus_alpha_hat * predicted_noise) / (sqrt_alpha_hat + 1e-8), -1.0, 1.0)

class SSIMLoss(nn.Module):
    def __init__(self, window_size=11):
        super().__init__()
        self.window_size = window_size
        gauss = torch.exp(torch.tensor([-((x - window_size // 2) ** 2) / (2 * 1.5**2) for x in range(window_size)]))
        gauss = gauss / gauss.sum()
        _2D = gauss.unsqueeze(1).mm(gauss.unsqueeze(1).t()).float().unsqueeze(0).unsqueeze(0)
        self.register_buffer("window", _2D)
    def forward(self, img1, img2):
        w = self.window.to(dtype=img1.dtype, device=img1.device)
        mu1 = F.conv2d(img1, w, padding=self.window_size // 2)
        mu2 = F.conv2d(img2, w, padding=self.window_size // 2)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        s1 = F.conv2d(img1 * img1, w, padding=self.window_size // 2) - mu1_sq
        s2 = F.conv2d(img2 * img2, w, padding=self.window_size // 2) - mu2_sq
        s12 = F.conv2d(img1 * img2, w, padding=self.window_size // 2) - mu1_mu2
        ssim_map = ((2 * mu1_mu2 + 0.0001) * (2 * s12 + 0.0009)) / ((mu1_sq + mu2_sq + 0.0001) * (s1 + s2 + 0.0009))
        return 1.0 - ssim_map.mean()

class FastSobelLoss(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).unsqueeze(0).unsqueeze(0)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).unsqueeze(0).unsqueeze(0)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)
    def forward(self, p, t):
        pgx, pgy = F.conv2d(p, self.kx, padding=1), F.conv2d(p, self.ky, padding=1)
        tgx, tgy = F.conv2d(t, self.kx, padding=1), F.conv2d(t, self.ky, padding=1)
        return F.l1_loss(torch.sqrt(pgx**2 + pgy**2 + 1e-8), torch.sqrt(tgx**2 + tgy**2 + 1e-8))

class AnatomyAwareHybridLoss(nn.Module):
    def __init__(self, lambda_diff=1.0, lambda_bone_l1=1.5, lambda_ssim=1.2, lambda_edge=0.8):
        super().__init__()
        self.lambda_diff = lambda_diff
        self.lambda_bone_l1 = lambda_bone_l1
        self.lambda_ssim = lambda_ssim
        self.lambda_edge = lambda_edge
        self.mse = nn.MSELoss()
        self.ssim_fn = SSIMLoss()
        self.edge_fn = FastSobelLoss()
    def forward(self, pred_noise, true_noise, pred_ct, real_ct, mask):
        l_diff = self.mse(pred_noise, true_noise)
        # Quadruple penalty on bone > 300 HU
        w = (1.0 + 4.0 * (real_ct > 0.04).float()) * (0.2 + 0.8 * mask)
        l_bone = torch.mean(w * torch.abs(pred_ct - real_ct))
        l_ssim = self.ssim_fn(pred_ct, real_ct)
        l_edge = self.edge_fn(pred_ct, real_ct)
        return self.lambda_diff * l_diff + self.lambda_bone_l1 * l_bone + self.lambda_ssim * l_ssim + self.lambda_edge * l_edge, l_ssim, l_edge, l_bone

scheduler = DDPMScheduler(timesteps=1000, device=device)
criterion = AnatomyAwareHybridLoss().to(device)
print("[OK] DDPMScheduler and AnatomyAwareHybridLoss loaded.")


In [ ]:
NUM_EPOCHS = 80
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

fixed_b = next(iter(val_loader))
fixed_mri = fixed_b['mri'][:1].to(device)
fixed_ct  = fixed_b['ct'][:1].to(device)
fixed_msk = fixed_b['mask'][:1].to(device)

best_val_loss = float('inf')
print("=" * 75)
print(f" STARTING PRODUCTION RE-TRAINING (80 EPOCHS | BONE-WEIGHTED + DDIM 50) ")
print(f" TARGET BENCHMARK: Test SSIM >= 0.8500 | Test PSNR >= 26 dB | Bone Dice >= 0.85 ")
print("=" * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    r_loss, r_ssim = 0.0, 0.0
    for batch in train_loader:
        mri, real_ct, mask = batch['mri'].to(device), batch['ct'].to(device), batch['mask'].to(device)
        t = torch.randint(0, scheduler.timesteps, (mri.size(0),), device=device).long()
        x_t, noise = scheduler.add_noise(real_ct, t)
        pred_noise = model(torch.cat([x_t, mri, mask], dim=1), t)
        pred_ct = predict_x0(x_t, pred_noise, t, scheduler)
        loss, l_ssim, _, _ = criterion(pred_noise, noise, pred_ct, real_ct, mask)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        r_loss += loss.item()
        r_ssim += l_ssim.item()

    lr_scheduler.step()
    cur_lr = optimizer.param_groups[0]['lr']

    # Validation
    model.eval()
    val_loss, val_ssim, val_bone = 0.0, 0.0, 0.0
    with torch.no_grad():
        for batch in val_loader:
            mri, real_ct, mask = batch['mri'].to(device), batch['ct'].to(device), batch['mask'].to(device)
            t = torch.randint(0, scheduler.timesteps, (mri.size(0),), device=device).long()
            x_t, noise = scheduler.add_noise(real_ct, t)
            pred_noise = model(torch.cat([x_t, mri, mask], dim=1), t)
            pred_ct = predict_x0(x_t, pred_noise, t, scheduler)
            v_loss, v_ssim, _, v_bone = criterion(pred_noise, noise, pred_ct, real_ct, mask)
            val_loss += v_loss.item()
            val_ssim += v_ssim.item()
            val_bone += v_bone.item()

    val_loss /= len(val_loader)
    val_ssim /= len(val_loader)
    val_bone /= len(val_loader)
    tr_loss = r_loss / len(train_loader)
    tr_ssim = r_ssim / len(train_loader)

    saved_str = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_cddpm_synthrad_sota.pth")
        saved_str = " ⭐ [BEST SAVED]"

    if epoch % 5 == 0 or epoch == 1 or saved_str != "":
        print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] (LR: {cur_lr:.1e}) | Train: {tr_loss:.4f} (SSIM-loss: {tr_ssim:.4f}) | Val: {val_loss:.4f} (SSIM-loss: {val_ssim:.4f}, Bone: {val_bone:.4f}){saved_str}")

    # Visual preview every 10 epochs using Fast DDIM 50-step sampler
    if epoch % 10 == 0 or epoch == 1:
        with torch.no_grad():
            syn_prev = scheduler.sample_ddim(model, fixed_mri, fixed_msk, ddim_steps=50)
        m_d = ((fixed_mri[0, 0] + 1.0) / 2.0).clamp(0, 1).cpu().numpy()
        s_d = ((syn_prev[0, 0]  + 1.0) / 2.0).clamp(0, 1).cpu().numpy()
        g_d = ((fixed_ct[0, 0]  + 1.0) / 2.0).clamp(0, 1).cpu().numpy()
        fig, axs = plt.subplots(1, 3, figsize=(10, 3.4))
        axs[0].imshow(m_d, cmap='gray'); axs[0].set_title(f"Val MRI (Ep {epoch})"); axs[0].axis('off')
        axs[1].imshow(s_d, cmap='gray'); axs[1].set_title(f"Synthetic CT (DDIM 50 | Ep {epoch})"); axs[1].axis('off')
        axs[2].imshow(g_d, cmap='gray'); axs[2].set_title("Ground Truth CT"); axs[2].axis('off')
        plt.tight_layout()
        plt.show()

torch.save(model.state_dict(), "final_cddpm_synthrad_sota.pth")
print("\n[OK] Training completed! Weights saved to 'best_cddpm_synthrad_sota.pth'")


In [ ]:
print("=" * 80)
print(f" EVALUATING FINAL MODEL ON 10% ISOLATED TEST PATIENTS ({len(test_p)} Patients | {len(te_m)} Slices)")
print("=" * 80)
model.load_state_dict(torch.load("best_cddpm_synthrad_sota.pth", map_location=device))
model.eval()

def compute_dice_and_hd95(real_bin, fake_bin):
    intersection = np.sum(real_bin & fake_bin)
    total_area = np.sum(real_bin) + np.sum(fake_bin)
    dice = (2.0 * intersection) / (total_area + 1e-8)
    coords_real = np.argwhere(real_bin)
    coords_fake = np.argwhere(fake_bin)
    if len(coords_real) == 0 or len(coords_fake) == 0:
        return float(dice), 0.0
    d1 = directed_hausdorff(coords_real, coords_fake)[0]
    d2 = directed_hausdorff(coords_fake, coords_real)[0]
    return float(dice), float(np.percentile([d1, d2], 95))

hu_min, hu_max = -1000.0, 1500.0
hu_range = hu_max - hu_min
ssims, psnrs, maes, mses, hu_maes, dices, hd95s = [], [], [], [], [], [], []
preview_samples = []

t0 = time.time()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating Test CTs"):
        mri, gt_ct, mask = batch['mri'].to(device), batch['ct'].to(device), batch['mask'].to(device)
        syn_ct = scheduler.sample_ddim(model, mri, mask, ddim_steps=50)

        s_np = ((syn_ct + 1.0) / 2.0).clamp(0, 1).cpu().numpy()
        g_np = ((gt_ct  + 1.0) / 2.0).clamp(0, 1).cpu().numpy()
        m_np = ((mri    + 1.0) / 2.0).clamp(0, 1).cpu().numpy()

        for i in range(mri.size(0)):
            s, g, m = s_np[i, 0], g_np[i, 0], m_np[i, 0]
            s_val = float(compute_ssim(g, s, data_range=1.0))
            p_val = float(compute_psnr(g, s, data_range=1.0))
            m_val = float(np.mean(np.abs(s - g)))
            mse_v = float(np.mean((s - g) ** 2))
            r_hu = g * hu_range + hu_min
            f_hu = s * hu_range + hu_min
            hu_v = float(np.mean(np.abs(r_hu - f_hu)))
            d_v, h_v = compute_dice_and_hd95(r_hu > 300.0, f_hu > 300.0)

            ssims.append(s_val); psnrs.append(p_val); maes.append(m_val)
            mses.append(mse_v); hu_maes.append(hu_v); dices.append(d_v); hd95s.append(h_v)

            if len(preview_samples) < 5:
                preview_samples.append({"mri": m, "syn": s, "gt": g, "ssim": s_val, "psnr": p_val, "hu": hu_v, "name": os.path.basename(batch['mri_path'][i])})

eval_time = time.time() - t0
report = {
    "Clinical Metric": ["SSIM (↑)", "PSNR (↑)", "MAE (↓)", "MSE (↓)", "HU-MAE (↓)", "Bone Dice (↑)", "Bone HD95 (↓)", "Latency"],
    "Target Benchmark": [">= 0.8500", ">= 26.0 dB", "<= 0.0350", "<= 0.0035", "<= 65.0 HU", ">= 0.8500", "<= 4.5 mm", "< 2.0 s/slice"],
    "Achieved SOTA Score": [
        f"{np.mean(ssims):.4f} ± {np.std(ssims):.4f}",
        f"{np.mean(psnrs):.2f} ± {np.std(psnrs):.2f} dB",
        f"{np.mean(maes):.4f} ± {np.std(maes):.4f}",
        f"{np.mean(mses):.6f} ± {np.std(mses):.6f}",
        f"{np.mean(hu_maes):.1f} ± {np.std(hu_maes):.1f} HU",
        f"{np.mean(dices):.4f} ± {np.std(dices):.4f}",
        f"{np.mean(hd95s):.2f} ± {np.std(hd95s):.2f} mm",
        f"{(eval_time / len(test_loader.dataset)):.2f} s/slice"
    ]
}

df_rep = pd.DataFrame(report)
print("\n" + "=" * 80)
print("     FINAL CLINICAL BENCHMARK REPORT: SYNTHRAD2023 TEST SET          ")
print("=" * 80)
print(df_rep.to_string(index=False))
print("=" * 80)
df_rep.to_csv("synthrad2023_sota_metrics.csv", index=False)

# Publication Plot
fig, axes = plt.subplots(4, len(preview_samples), figsize=(len(preview_samples) * 3.4, 12))
for col, item in enumerate(preview_samples):
    err = np.abs(item['syn'] - item['gt'])
    axes[0, col].imshow(item['mri'], cmap='gray'); axes[0, col].set_title(f"Input MRI\n{item['name']}", fontsize=8); axes[0, col].axis('off')
    axes[1, col].imshow(item['syn'], cmap='gray'); axes[1, col].set_title(f"Synthetic CT (DDIM)\nSSIM: {item['ssim']:.4f} | {item['psnr']:.1f} dB", fontsize=9, fontweight='bold', color='navy'); axes[1, col].axis('off')
    axes[2, col].imshow(item['gt'], cmap='gray'); axes[2, col].set_title(f"Ground Truth CT\nHU-MAE: {item['hu']:.1f} HU", fontsize=9); axes[2, col].axis('off')
    im = axes[3, col].imshow(err, cmap='inferno', vmin=0, vmax=0.30); axes[3, col].set_title("Absolute Error Map", fontsize=9); axes[3, col].axis('off')

plt.tight_layout()
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5, pad=0.03, label='|Synthetic - GT| Error')
plt.savefig("synthrad2023_sota_visual_grid.png", dpi=300, bbox_inches='tight')
plt.show()
print("[OK] Saved publication plot to 'synthrad2023_sota_visual_grid.png'")
